# Análisis Descriptivo de la Cohorte

Este notebook genera las figuras y tablas descriptivas de la cohorte final utilizada en la tesis. Todos los datos se procesan con los mismos módulos del pipeline principal (`src/`) para garantizar consistencia en la normalización de IDs, construcción de la cohorte y filtros aplicados.

Las figuras se guardan directamente en el directorio de imágenes del capítulo de datos (`Latex/05_datos_cohorte/images/`) para ser incluidas en el documento LaTeX.

### Configuración del entorno

Resolución de la raíz del proyecto, imports de los módulos del pipeline, configuración de matplotlib (DPI, tamaño de fuente, etc.) y definición de la función auxiliar `save_fig` para guardar las figuras directamente en el directorio LaTeX del capítulo de datos.

In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "src").exists():
    ROOT = _cwd
elif (_cwd.parent / "src").exists():
    ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {_cwd}")

REPO = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Code root:", ROOT)
print("Repo root:", REPO)

Code root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Code
Repo root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci


### Imports, paths y configuración de matplotlib

Se importan las dependencias, se configuran las rutas a los datos y el estilo global de las figuras (DPI, tamaño de fuente, ejes sin bordes superiores/derechos). La función `save_fig` guarda cada figura directamente en el directorio de imágenes del capítulo LaTeX.

In [2]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from src.config import Paths, ExperimentConfig
from src.cohort import build_final_cohort_df
from src.data_io import (
    read_t1w_csv,
    list_fc_files,
    extract_fc_record_id_from_filename,
    _load_fc_matrix,
)

paths = Paths(
    excel_path=REPO / "Data" / "datos-redlat.xlsx",
    fc_folder=REPO / "Data" / "fc_mats",
    t1w_csv_path=REPO / "Data" / "Redlat_VGM_AAL_.csv",
    out_dir=REPO / "Outputs",
)

cfg = ExperimentConfig(seed=42, diagnoses_to_use=("CN", "AD", "FTD"))

IMGDIR = REPO / "Latex" / "05_datos_cohorte" / "images"
IMGDIR.mkdir(parents=True, exist_ok=True)

np.random.seed(cfg.seed)

mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def save_fig(fname: str):
    plt.tight_layout()
    plt.savefig(IMGDIR / fname, bbox_inches="tight", pad_inches=0.05)
    plt.close()
    print("Saved:", IMGDIR / fname)

print("Output images:", IMGDIR)

Output images: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images


## Construcción de la cohorte

Se utiliza la misma función del pipeline (`build_final_cohort_df`) que aplica la normalización de IDs, el colapso de duplicados T1w por promedio, y la intersección FC ∩ Metadata ∩ T1w.

In [3]:
cohort = build_final_cohort_df(
    excel_path=paths.excel_path,
    fc_folder=paths.fc_folder,
    t1w_csv_path=paths.t1w_csv_path,
    diagnoses_to_use=cfg.diagnoses_to_use,
)

# Compute mean gray matter volume per subject from T1w features
t1_cols = [c for c in cohort.columns if c.startswith("t1_")]
cohort["gm_mean"] = cohort[t1_cols].mean(axis=1)

print(f"Cohort: {len(cohort)} subjects, {len(t1_cols)} T1w features")
print(f"Diagnoses: {cohort['diagnosis'].value_counts().to_dict()}")
print(f"Sex: {cohort['sex'].value_counts().to_dict()}")
print(f"Age: {cohort['age'].min():.1f} - {cohort['age'].max():.1f} years")
cohort[["record_id", "age", "sex", "diagnosis"]].head()

Cohort: 1245 subjects, 116 T1w features
Diagnoses: {'CN': 526, 'AD': 422, 'FTD': 297}
Sex: {'Male': 767, 'Female': 478}
Age: 18.0 - 98.0 years


,record_id,age,sex,diagnosis
0,AF025,75.0,Female,CN
1,AF036,74.0,Female,AD
2,AF037,71.0,Male,FTD
3,AF063,73.0,Female,CN
4,AF065,65.0,Male,CN


## Distribución de edad

In [4]:
plt.figure(figsize=(6.6, 4.2))
plt.hist(cohort["age"].values, bins=30)
plt.xlabel("Edad (años)")
plt.ylabel("Cantidad de sujetos")
save_fig("age_hist.png")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/age_hist.png


## Edad por diagnóstico

In [5]:
diag_order = [d for d in cfg.diagnoses_to_use if d in cohort["diagnosis"].unique()]
data = [cohort.loc[cohort["diagnosis"] == d, "age"].dropna().values for d in diag_order]

plt.figure(figsize=(6.8, 4.2))
plt.boxplot(data, labels=diag_order, showfliers=False)
plt.xlabel("Diagnóstico clínico")
plt.ylabel("Edad (años)")
save_fig("age_by_diag_box.png")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/age_by_diag_box.png


/tmp/ipykernel_763249/1564694688.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=diag_order, showfliers=False)


## Edad por diagnóstico y sexo

In [6]:
sexes = [s for s in ["Female", "Male"] if s in cohort["sex"].unique()]
labels, series = [], []

for d in diag_order:
    for s in sexes:
        vals = cohort.loc[(cohort["diagnosis"] == d) & (cohort["sex"] == s), "age"].dropna().values
        if len(vals) == 0:
            continue
        series.append(vals)
        labels.append(f"{d}\n{s[0]}")

plt.figure(figsize=(8.6, 4.6))
plt.boxplot(series, labels=labels, showfliers=False)
plt.xlabel("Diagnóstico / Sexo")
plt.ylabel("Edad (años)")
save_fig("age_by_diag_sex_box.png")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/age_by_diag_sex_box.png


/tmp/ipykernel_763249/3672292837.py:13: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(series, labels=labels, showfliers=False)


## Distribución de sexo por diagnóstico

In [7]:
ct = pd.crosstab(cohort["diagnosis"], cohort["sex"])
order = [d for d in diag_order if d in ct.index]
ct = ct.loc[order]
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100.0

plt.figure(figsize=(7.2, 4.8))
bottom = np.zeros(len(ct_pct), dtype=float)
x = np.arange(len(ct_pct.index))

for col in ct_pct.columns:
    vals = ct_pct[col].values
    plt.bar(x, vals, bottom=bottom, label=str(col))
    bottom += vals

plt.xticks(x, ct_pct.index.astype(str), rotation=20, ha="right")
plt.xlabel("Diagnóstico clínico")
plt.ylabel("Porcentaje de sujetos (%)")
plt.ylim(0, 100)
plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.18), ncols=2, frameon=False)
save_fig("sex_by_diag_stacked.png")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/sex_by_diag_stacked.png


## Correlación entre edad y materia gris (T1w)

In [9]:
def corr_report(x, y):
    """Pearson and Spearman correlations."""
    x0, y0 = x - np.nanmean(x), y - np.nanmean(y)
    pear = float(np.nansum(x0 * y0) / np.sqrt(np.nansum(x0**2) * np.nansum(y0**2) + 1e-12))
    xr = pd.Series(x).rank().to_numpy()
    yr = pd.Series(y).rank().to_numpy()
    xr0, yr0 = xr - np.nanmean(xr), yr - np.nanmean(yr)
    spear = float(np.nansum(xr0 * yr0) / np.sqrt(np.nansum(xr0**2) * np.nansum(yr0**2) + 1e-12))
    return pear, spear

tmp = cohort[["age", "gm_mean"]].dropna()
x_age = tmp["age"].to_numpy(float)
y_gm = tmp["gm_mean"].to_numpy(float)
pear, spear = corr_report(x_age, y_gm)

plt.figure(figsize=(6.6, 4.8))
plt.scatter(x_age, y_gm, s=12, alpha=0.55)
m, b = np.polyfit(x_age, y_gm, 1)
xs = np.linspace(x_age.min(), x_age.max(), 100)
plt.plot(xs, m * xs + b, linewidth=2.0)
plt.xlabel("Edad (años)")
plt.ylabel("Media regional de materia gris (T1w)")
plt.title(f"Edad vs materia gris (Pearson r={pear:.2f}, Spearman ρ={spear:.2f})")
plt.grid(True, alpha=0.2)
save_fig("corr_age_gmmean.png")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/corr_age_gmmean.png


## Correlación entre edad y conectividad funcional global

In [10]:
# Build record_id -> path mapping using src.data_io
rid_to_path = {}
for p in list_fc_files(paths.fc_folder):
    rid = extract_fc_record_id_from_filename(p)
    if rid:
        rid_to_path[rid] = p

def fc_global_mean_strength(mat):
    """Mean absolute off-diagonal correlation."""
    n = mat.shape[0]
    mask = ~np.eye(n, dtype=bool)
    return float(np.nanmean(np.abs(mat[mask])))

fc_means, ages = [], []
for rid, age in zip(cohort["record_id"].values, cohort["age"].values):
    p = rid_to_path.get(rid)
    if p is None:
        continue
    try:
        mat = _load_fc_matrix(p)
        fc_means.append(fc_global_mean_strength(mat))
        ages.append(float(age))
    except Exception:
        continue

x_age_fc = np.asarray(ages, float)
y_fc = np.asarray(fc_means, float)
pear, spear = corr_report(x_age_fc, y_fc)

plt.figure(figsize=(6.6, 4.8))
plt.scatter(x_age_fc, y_fc, s=12, alpha=0.55)
m, b = np.polyfit(x_age_fc, y_fc, 1)
xs = np.linspace(x_age_fc.min(), x_age_fc.max(), 100)
plt.plot(xs, m * xs + b, linewidth=2.0)
plt.xlabel("Edad (años)")
plt.ylabel("Promedio |corr| fuera de diagonal (FC)")
plt.title(f"Edad vs conectividad global (Pearson r={pear:.2f}, Spearman ρ={spear:.2f})")
plt.grid(True, alpha=0.2)
save_fig("corr_age_fcmean.png")

print(f"FC global metric computed for {len(ages)} subjects")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/corr_age_fcmean.png
FC global metric computed for 1245 subjects


## Tablas LaTeX

Se generan tablas en formato LaTeX para inclusión directa en el documento de tesis.

In [ ]:
def save_latex_table(df, path, caption, label, index=False, colfmt=None):
    """Write a DataFrame as a standalone LaTeX table."""
    df2 = df.copy()
    df2.index.name = ""
    tex = df2.to_latex(
        index=index, escape=False, float_format="%.1f",
        column_format=colfmt, index_names=False,
    )
    tex = (
        "\\begin{table}[H]\n\\centering\n"
        + tex
        + "\\caption{" + caption + "}\n"
        + "\\label{" + label + "}\n"
        + "\\end{table}\n"
    )
    with open(path, "w", encoding="utf-8") as f:
        f.write(tex)
    print("Saved:", path)

### Generación de las tres tablas

Se generan las tablas de distribución diagnóstica, distribución por sexo y resumen de edad, cada una como archivo `.tex` independiente.

In [ ]:
# Diagnosis distribution
diag_counts = cohort["diagnosis"].value_counts()
diag_counts = diag_counts.loc[[d for d in diag_order if d in diag_counts.index]]
diag_pct = (diag_counts / diag_counts.sum() * 100).round(1)

diag_tab = pd.DataFrame({"Conteo": diag_counts.astype(int), "Porcentaje": diag_pct})
save_latex_table(
    diag_tab, IMGDIR / "diag_table.tex",
    caption=f"Distribución diagnóstica en la cohorte final (N={len(cohort)}).",
    label="tab:diag_dist", index=True, colfmt="lrr",
)

# Sex distribution
sex_counts = cohort["sex"].value_counts()
sex_pct = (sex_counts / sex_counts.sum() * 100).round(1)

sex_tab = pd.DataFrame({"Conteo": sex_counts.astype(int), "Porcentaje": sex_pct})
save_latex_table(
    sex_tab, IMGDIR / "sex_table.tex",
    caption=f"Distribución por sexo en la cohorte final (N={len(cohort)}).",
    label="tab:sex_dist", index=True, colfmt="lrr",
)

# Age summary
q1 = cohort["age"].quantile(0.25)
q3 = cohort["age"].quantile(0.75)
age_summary = pd.DataFrame([{
    "N": int(cohort["age"].notna().sum()),
    "Media": round(float(cohort["age"].mean()), 1),
    "DE": round(float(cohort["age"].std()), 1),
    "Mediana": round(float(cohort["age"].median()), 1),
    "IQR": round(float(q3 - q1), 1),
    "Min--Max": f"{cohort['age'].min():.1f}--{cohort['age'].max():.1f}",
}])

save_latex_table(
    age_summary, IMGDIR / "age_summary_table.tex",
    caption="Resumen descriptivo de la edad en la cohorte final.",
    label="tab:age_summary", index=False, colfmt="rrrrrl",
)

print("All tables generated.")

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/diag_table.tex
Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/sex_table.tex
Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Latex/05_datos_cohorte/images/age_summary_table.tex
All tables generated.


### Generación de tablas LaTeX

Se generan tres tablas en formato LaTeX: distribución diagnóstica, distribución por sexo, y resumen estadístico de la edad. Cada tabla se guarda como archivo `.tex` para inclusión directa en el documento.